# Phase 06A.00 — Frozen data audit and protocol lock

Audits the canonical 1,192/298 group-safe split, validates the preregistered validation checksum, decodes videos, checks exact duplicate files, and writes the immutable Phase 06A protocol. Legacy 319-row artifacts are never accepted as canonical evidence.

In [1]:
import json, os, sys
from pathlib import Path

PROJECT_ROOT = Path('/workspace/RoadBuddy')
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))
os.chdir(PROJECT_ROOT)
from roadbuddy_common import *
from phase06a_common import *
seed_everything(SEED)

## Configuration
Set `RUN_SCOPE='full'` only on the canonical server dataset. Smoke artifacts cannot unlock downstream full experiments.

In [2]:
RUN_SCOPE = 'smoke'  # explicit: smoke | full
SMOKE_VIDEO_LIMIT = 6
TRAIN_CSV = PROJECT_ROOT / 'data/splits/phase01/train.csv'
VALIDATION_CSV = PROJECT_ROOT / 'data/splits/phase01/validation.csv'
VALIDATION_IDS = PROJECT_ROOT / 'data/splits/phase01/validation_sample_ids.json'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/phase06a/data_audit' / RUN_SCOPE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
protocol = Phase06AProtocol(run_scope=RUN_SCOPE)
assert all(path.is_file() for path in [TRAIN_CSV, VALIDATION_CSV, VALIDATION_IDS]), 'Canonical split artifacts are missing'

## Membership and leakage gates

In [3]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VALIDATION_CSV)
frozen_ids = json.loads(VALIDATION_IDS.read_text(encoding='utf-8'))
ids_file_hash = sha256_file(VALIDATION_IDS)
split_report = validate_canonical_split(train_df, val_df, frozen_ids, run_scope=RUN_SCOPE, expected_ids_file_sha256=ids_file_hash)
split_report['raw_labeled_rows'] = len(train_df) + len(val_df)
if RUN_SCOPE == 'full': assert split_report['raw_labeled_rows'] == EXPECTED_RAW_ROWS
display(split_report)

{'run_scope': 'smoke',
 'train_rows': 1192,
 'validation_rows': 298,
 'train_groups': 439,
 'validation_groups': 110,
 'sample_overlap': 0,
 'group_overlap': 0,
 'validation_membership_hash': '14c3f07a5c957d662c3acaf104cd3ac6d232e0d44cea86d6df5fbdf343754be7',
 'validation_ids_file_sha256': 'dbfa2d337f56bd681df70206fa8cee83d743c6743a3493619083d03b804145c5',
 'raw_labeled_rows': 1490}

## Decode and exact-duplicate audit
Full scope checks every unique video. Smoke scope checks a deterministic prefix and is never marked PASS.

In [4]:
all_video_rows = pd.concat([train_df.assign(split='train'), val_df.assign(split='validation')], ignore_index=True)
unique_paths = sorted(all_video_rows.video_path.astype(str).unique())
selected_paths = unique_paths if RUN_SCOPE == 'full' else unique_paths[:SMOKE_VIDEO_LIMIT]
decode_records = []
for video_path in selected_paths:
    record = {'video_path': video_path, 'exists': Path(video_path).is_file()}
    try:
        record.update(probe_video(video_path)); record['decodes'] = True; record['error'] = ''
    except Exception as error:
        record.update({'decodes': False, 'error': repr(error)})
    decode_records.append(record)
decode_report = pd.DataFrame(decode_records)
decode_report.to_csv(OUTPUT_DIR / 'video_decode_report.csv', index=False)
assert decode_report.decodes.all(), 'At least one audited video failed to decode'

hash_records = []
for video_path in selected_paths:
    path = Path(video_path)
    hash_records.append({'video_path': video_path, 'sha256': sha256_file(path) if path.is_file() else None})
hash_frame = pd.DataFrame(hash_records)
path_splits = all_video_rows.groupby('video_path').split.agg(lambda values: sorted(set(values))).to_dict()
duplicates = []
for digest, group in hash_frame.dropna().groupby('sha256'):
    paths = group.video_path.tolist()
    if len(paths) > 1:
        duplicates.append({'sha256': digest, 'video_paths': json.dumps(paths), 'splits': json.dumps([path_splits.get(path, []) for path in paths]), 'cross_split': len(set(sum([path_splits.get(path, []) for path in paths], []))) > 1})
duplicate_report = pd.DataFrame(duplicates, columns=['sha256','video_paths','splits','cross_split'])
duplicate_report.to_csv(OUTPUT_DIR / 'duplicate_video_report.csv', index=False)
if len(duplicate_report): assert not duplicate_report.cross_split.astype(bool).any(), 'Exact duplicate video leakage detected'

## Lock artifacts and status

In [5]:
audit = {**split_report, 'decoded_videos': len(decode_report), 'decode_failures': int((~decode_report.decodes).sum()), 'exact_duplicate_sets': len(duplicate_report), 'near_duplicate_audit': 'not_implemented_manual_followup'}
save_json(OUTPUT_DIR / 'dataset_audit.json', audit)
save_json(OUTPUT_DIR / 'phase06a_protocol.json', protocol.to_dict())
status = {'phase': '06A.00', 'run_scope': RUN_SCOPE, 'status': 'PASS' if RUN_SCOPE == 'full' else 'smoke_complete', 'hard_gates_passed': RUN_SCOPE == 'full', 'validation_ids_sha256': ids_file_hash}
save_json(OUTPUT_DIR / 'PHASE06A_00_STATUS.json', status)
display(status)

{'phase': '06A.00',
 'run_scope': 'smoke',
 'status': 'smoke_complete',
 'hard_gates_passed': False,
 'validation_ids_sha256': 'dbfa2d337f56bd681df70206fa8cee83d743c6743a3493619083d03b804145c5'}

## Interpretation constraint
Only a full-scope `PASS` status may unlock the remaining full Phase 06A notebooks. This notebook produces no model-quality result.